In [ ]:
!pip install evaluate
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.4 MB/s eta 0:00:00


In [ ]:
!pip install sacrebleu

In [ ]:

import pandas as pd
import numpy as np
import torch
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from datasets import Dataset
import evaluate

In [ ]:
df = pd.read_csv('NLP_QA_MT.csv')
df = df[['QuestionText_QA', 'Answer_QA']].dropna().reset_index(drop=True)

def clean_column(text):
    try:
        tokens = ast.literal_eval(text)
        if isinstance(tokens, list):
            return ' '.join(tokens)
    except:
        pass
    return str(text)

df['QuestionText_QA'] = df['QuestionText_QA'].apply(clean_column)
df['Answer_QA']       = df['Answer_QA'].apply(clean_column)

print(f'Dataset size: {len(df)}')
print(df.head(3))

Dataset size: 4645
                                     QuestionText_QA  \
0  االتمرينات الرياضيه اليوميه تساعد في تحسين الص...   
1                               اتصعد الشمس من الشرق   
2             اتعرف البكتيريا بانها كاينات حيه دقيقه   

                                           Answer_QA  
0  التمرينات الرياضيه اليوميه تساهم بشكل كبير في ...  
1                                الشمس تصعد من الشرق  
2              البكتيريا تعرف بانها كاينات حيه دقيقه  


In [ ]:
#Train Test Split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

train_hf = Dataset.from_pandas(train_df)
val_hf   = Dataset.from_pandas(test_df)

print(f'Train: {len(train_df)} | Test: {len(test_df)}')

Train: 3716 | Test: 929


# **T5**

In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer
)

model_name = 'google/mt5-small'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.gradient_checkpointing_enable()

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
def preprocess_data(examples):
    questions = [
        'أجب على السؤال: ' + q
        for q in examples['QuestionText_QA']
    ]
    answers = examples['Answer_QA']

    inputs = tokenizer(questions, truncation=True, max_length=128)
    labels = tokenizer(answers,   truncation=True, max_length=128)

    labels_ids = [
        [-100 if token == tokenizer.pad_token_id else token for token in label]
        for label in labels['input_ids']
    ]

    inputs['labels'] = labels_ids
    return inputs

In [ ]:
tokenized_train = train_hf.map(preprocess_data, batched=True)
tokenized_val   = val_hf.map(preprocess_data,   batched=True)

print(tokenized_train[0])

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

{'QuestionText_QA': 'ماذا تعني لك الموسيقي', 'Answer_QA': 'الموسيقي تعني لي وسيله للتعبير عن المشاعر والاسترخاء والاستمتاع', 'input_ids': [1021, 18861, 259, 942, 402, 42818, 1463, 267, 1415, 4216, 4355, 4271, 18764, 259, 166289, 96943, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [259, 166289, 96943, 4355, 4271, 12073, 83731, 2869, 2008, 102825, 4525, 1698, 1105, 31534, 115584, 13567, 2216, 115584, 15352, 21128, 1]}


In [ ]:
print(df["Answer_QA"].isnull().sum())
print((df["Answer_QA"]=="").sum())

0
0


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir='./mt5_qa',
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    predict_with_generate=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    fp16=False,
    load_best_model_at_end=True,
    report_to='none'
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,3.805366
2,30.196693,3.495175
3,21.039773,3.424501


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=1395, training_loss=23.920675123207886, metrics={'train_runtime': 24437.5482, 'train_samples_per_second': 0.456, 'train_steps_per_second': 0.057, 'total_flos': 282443213783040.0, 'train_loss': 23.920675123207886, 'epoch': 3.0})

In [ ]:
def answer_mt5(question):
    input_text = 'أجب على السؤال: ' + question
    inputs = tokenizer(
        input_text,
        return_tensors='pt',
        truncation=True,
        max_length=128
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=64, num_beams=4)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
bleu = evaluate.load('sacrebleu')

mt5_predictions = []
mt5_references  = []

for _, row in test_df.iterrows():
    pred = answer_mt5(row['QuestionText_QA'])
    mt5_predictions.append(pred)
    mt5_references.append([row['Answer_QA']])

result_mt5 = bleu.compute(predictions=mt5_predictions, references=mt5_references)
print('mT5 BLEU Score:', result_mt5['score'])

mT5 BLEU Score: 8.856918567511936


# **QWEN**

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

model_name = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForCausalLM.from_pretrained(model_name, device_map='auto')

tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def format_data_qwen(example):
    q = example['QuestionText_QA']
    a = example['Answer_QA']
    text = f'سؤال: {q}\nجواب: {a}'
    return {'text': text}

train_qwen = train_hf.map(format_data_qwen)
val_qwen   = val_hf.map(format_data_qwen)

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

In [ ]:
def tokenize_data(example):
    return tokenizer(example['text'], truncation=True, max_length=256)

train_qwen = train_qwen.map(tokenize_data, batched=True, remove_columns=train_qwen.column_names)
val_qwen   = val_qwen.map(tokenize_data,   batched=True, remove_columns=val_qwen.column_names)

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install -U peft transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 36.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from peft import LoraConfig, get_peft_model

In [ ]:
lora_config = LoraConfig(
    r=8,
    target_modules=['q_proj', 'v_proj'],
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./qwen_qa',
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    num_train_epochs=2,
    eval_strategy='epoch',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_qwen,
    eval_dataset=val_qwen,
    data_collator=data_collator
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,1.487128,1.512534
2,1.380941,1.465820


TrainOutput(global_step=7432, training_loss=1.4880673313038213, metrics={'train_runtime': 2879.042, 'train_samples_per_second': 2.581, 'train_steps_per_second': 2.581, 'total_flos': 3501786016229376.0, 'train_loss': 1.4880673313038213, 'epoch': 2.0})

In [ ]:
def answer_qwen(question):
    prompt = f'سؤال: {question}\nجواب:'
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split('جواب:')[-1].strip()

In [ ]:
bleu = evaluate.load('sacrebleu')

qwen_predictions = []
qwen_references  = []

for _, row in test_df.iterrows():
    pred = answer_qwen(row['QuestionText_QA'])
    qwen_predictions.append(pred)
    qwen_references.append([row['Answer_QA']])

result_qwen = bleu.compute(predictions=qwen_predictions, references=qwen_references)
print('Qwen BLEU Score:', result_qwen['score'])

Qwen BLEU Score: 9.17065531772371


# **GPT**

In [ ]:
# Free Qwen from memory
del model, trainer
torch.cuda.empty_cache()

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

model_name = 'aubmindlab/aragpt2-base'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForCausalLM.from_pretrained(model_name).to('cuda')

tokenizer.pad_token = tokenizer.eos_token

print('araGPT2 loaded.')

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: aubmindlab/aragpt2-base
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


araGPT2 loaded.


In [ ]:
def format_data_gpt(example):
    q = example['QuestionText_QA']
    a = example['Answer_QA']
    text = f'سؤال: {q}\nجواب: {a}'
    return {'text': text}

train_gpt = train_hf.map(format_data_gpt)
val_gpt   = val_hf.map(format_data_gpt)

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

In [ ]:
def tokenize_gpt(example):
    return tokenizer(example['text'], truncation=True, max_length=256)

train_gpt = train_gpt.map(tokenize_gpt, batched=True, remove_columns=train_gpt.column_names)
val_gpt   = val_gpt.map(tokenize_gpt,   batched=True, remove_columns=val_gpt.column_names)

Map:   0%|          | 0/3716 [00:00<?, ? examples/s]

Map:   0%|          | 0/929 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./gpt_qa',
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    num_train_epochs=3,
    eval_strategy='epoch',
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_gpt,
    eval_dataset=val_gpt,
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.937961,2.703125
2,2.263804,2.492441
3,1.520653,2.524202


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2787, training_loss=2.4344592250116333, metrics={'train_runtime': 465.9109, 'train_samples_per_second': 23.927, 'train_steps_per_second': 5.982, 'total_flos': 289374801408000.0, 'train_loss': 2.4344592250116333, 'epoch': 3.0})

In [ ]:
def answer_gpt(question):
    prompt = f'سؤال: {question}\nجواب:'
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=128).to('cuda')

    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split('جواب:')[-1].strip()

In [ ]:
bleu = evaluate.load('sacrebleu')

gpt_predictions = []
gpt_references  = []

for _, row in test_df.iterrows():
    pred = answer_gpt(row['QuestionText_QA'])
    gpt_predictions.append(pred)
    gpt_references.append([row['Answer_QA']])

result_gpt = bleu.compute(predictions=gpt_predictions, references=gpt_references)
print('araGPT2 BLEU Score:', result_gpt['score'])

araGPT2 BLEU Score: 7.643766936588366
